# Benchmark de localización
Compara YOLO26n, YOLO26s y YOLO11n con el mismo dataset, split y configuración.


In [ ]:
!pip install -q -U ultralytics pandas


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO

DATA_YAML = "/kaggle/input/tu-dataset/data.yaml"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PROJECT = "/kaggle/working/yolo_benchmark"
CANDIDATES = ["yolo26n.pt", "yolo26s.pt", "yolo11n.pt"]

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
rows = []
for weights in CANDIDATES:
    run_name = Path(weights).stem
    model = YOLO(weights)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, project=PROJECT, name=run_name, exist_ok=True)
    metrics = model.val(data=DATA_YAML, split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE)
    rows.append({
        "model": run_name,
        "map50_95": float(metrics.box.map),
        "map50": float(metrics.box.map50),
        "map75": float(metrics.box.map75),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    })

results = pd.DataFrame(rows).sort_values("map50_95", ascending=False).reset_index(drop=True)
results


In [ ]:
results.to_csv("/kaggle/working/yolo_benchmark.csv", index=False)
best_name = results.iloc[0]["model"]
best_weights = Path(PROJECT) / best_name / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
best_model.export(format="onnx", imgsz=IMGSZ)
print(best_name)
print(best_weights)
